<div class="alert alert-block alert-info">

### Imports & Connection
</div>

In [15]:
import os
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import datetime as datetime
import plotly.graph_objects as go

from scipy import stats
from pyhive import presto
from IPython.display import display
from datetime import datetime, timedelta
# from scipy.stats import gaussian_kde

import warnings
warnings.filterwarnings('ignore')

In [16]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

In [17]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [18]:
print('Notebook dir - ', os.getcwd())
local_extract = '/Users/E2074/local-datasets/customer/ads-monetisation/monet-benchmark'
print('Data dir - ', local_extract)

Notebook dir -  /Users/E2074/analytics/customer/Ads-Monetisation/monet-benchmark
Data dir -  /Users/E2074/local-datasets/customer/ads-monetisation/monet-benchmark


<div class="alert alert-block alert-info">
    
## Datasets & Parameter
</div>

In [19]:
## Parameter 
start_date = '20241128'
end_date = '20241204'

city_list = [  'Bangalore', 'Hyderabad', 'Delhi', 'Chennai', 'Ahmedabad',
                'Kolkata', 'Lucknow', 'Jaipur', 'Pune', 'Chandigarh', 
                'Vijayawada', 'Mumbai', 'Indore', 'Tirupati', 'Patna', 
                 'Coimbatore', 'Guwahati', 'Vishakapatnam']

<div class="alert alert-block alert-warning">
    
#### Required Function 
</div>

In [20]:
def func_ad_response(date,segment_date, city):
    
    
    base_query = f"""


    with response AS (
        
            SELECT
                res.yyyymmdd as yyyymmdd,
                res.city as city,
                res.os as os,
                res.responsetype as responsetype,  
                res.userid as userid,
                res.slotname as slotname, 
                res.format as format, 
                res.responsestatus as responsestatus,
                -- res.isprefetched as isprefetched,
                coalesce(res.aduuid, res.adid) as response_id,
                REPLACE(REGEXP_EXTRACT(metadata1, 'title=([^,]*)'), 'title=', '') AS title,
                REPLACE(REGEXP_EXTRACT(metadata1, 'advertiserName=([^,]*)'), 'advertiserName=', '') AS advertiserName,
                REPLACE(REGEXP_EXTRACT(metadata1, 'body=([^,]*)'), 'body=', '') AS body
            FROM 
                canonical.iceberg_log_telemetry_ads_impressions_immutable res
            WHERE 
                res.yyyymmdd = '{date}'
                AND res.eventName = 'Ad_Response'
                AND res.responsetype IN ('GAMBanner', 'ONLINE_SALES', 'System Banner')
                AND res.city = '{city}'
                -- AND NOT regexp_like(res.format, 'Arrived|OTW|Started|nativeBanner')
                AND (res.responsestatus = 'true' OR res.responsestatus IS NULL)

                -- AND quarter_hour = '1115'
        ),

        segment as (

        SELECT
            customer_id,
            taxi_ltr_segment,
            taxi_retention_segments taxi_retention_segment,
            taxi_regularity_segment taxi_rr_regularity_segment,
            -- taxi_intent_segment taxi_rr_intent_segment,
            taxi_fe_regularity_segment
            -- taxi_fe_intent_segment
        FROM 
            datasets.iallocator_customer_segments
        WHERE 
            run_date = '{segment_date}'
            AND taxi_recency_segment != 'INACTIVE'
        )
        
        SELECT
            response.yyyymmdd as yyyymmdd,
            response.city as city,
            response.os as os,
            response.responsetype as responsetype,  
            response.userid as userid,
            response.slotname as slotname, 
            response.format as format, 
            response.responsestatus as responsestatus,
            -- response.isprefetched as isprefetched,
            
            taxi_ltr_segment,
            taxi_retention_segment,
            taxi_rr_regularity_segment,
            -- taxi_rr_intent_segment,
            taxi_fe_regularity_segment,
            -- taxi_fe_intent_segment,

            response_id,
            
            title,
            advertiserName,
            body
            
        FROM    
            response
        LEFT JOIN 
            segment as segment
            ON response.userid = segment.customer_id

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base



In [21]:
def func_ad_impression(date, city):
    
    
    base_query = f"""
    
        SELECT
            DISTINCT
            coalesce(imp.aduuid, imp.adid) as impression_id,
            imp.userid as userid,
            imp.yyyymmdd as yyyymmdd
        FROM 
            canonical.iceberg_log_telemetry_ads_impressions_immutable as imp
        WHERE 
            imp.yyyymmdd = '{date}'
            AND imp.eventName = 'Ad_Impression'
            AND imp.responsetype IN ('GAMBanner', 'ONLINE_SALES', 'System Banner')
            AND imp.city = '{city}'
            -- AND quarter_hour = '1115'

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base

In [22]:
def func_ad_viewable_impression(date, city):
    
    
    base_query = f"""
    
        SELECT
            DISTINCT
            coalesce(view_imp.aduuid, view_imp.adid) as viewable_impression_id,
            view_imp.userid as userid,
            view_imp.yyyymmdd as yyyymmdd
        FROM 
            canonical.iceberg_log_telemetry_ads_impressions_immutable view_imp
        WHERE 
            view_imp.yyyymmdd = '{date}'
            AND view_imp.eventName = 'Ad_Viewable_Impression'
            AND view_imp.responsetype IN ('GAMBanner', 'ONLINE_SALES', 'System Banner')
            AND view_imp.city = '{city}'
            -- AND quarter_hour = '1115'

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base

In [23]:
def func_ad_click(date, city):
    
    
    base_query = f"""

            
        SELECT
            DISTINCT
            coalesce(clk.aduuid, clk.adid) as click_id,
            clk.userid as userid,
            clk.yyyymmdd as yyyymmdd
        FROM 
            canonical.iceberg_log_telemetry_ads_impressions_immutable as clk
        WHERE 
            clk.yyyymmdd = '{date}'
            AND clk.eventName = 'Ad_Click'
            AND clk.responsetype IN ('GAMBanner', 'ONLINE_SALES', 'System Banner')
            AND clk.city = '{city}'
            -- AND quarter_hour = '1115'

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base

In [24]:
def func_pull_ads_data(start_date, end_date, city_list):
    
    date_list = pd.date_range(start=start_date, end=end_date).strftime('%Y%m%d').tolist()
    formatted_end_date = datetime.strptime(end_date, '%Y%m%d').strftime('%Y-%m-%d')
    
    for date in date_list:

        for city in city_list:
            
            print(date, city)
            df_response_data_dump = func_ad_response(date, formatted_end_date, city)
            print('response')
            # display(df_response_data_dump.head(5))
            
            df_impression_data_dump = func_ad_impression(date, city)
            print('impression')
            # display(df_impression_data_dump.head(5))
            
            df_viewable_impression_data_dump = func_ad_viewable_impression(date, city)
            print('viewability')
            # display(df_viewable_impression_data_dump.head(5))
            
            df_click_data_dump = func_ad_click(date, city)
            print('click')
            # display(df_click_data_dump.head(5))
            
            merge1 = pd.merge(df_response_data_dump, df_impression_data_dump, 
                              how='left', 
                              left_on=['yyyymmdd', 'userid', 'response_id'],
                              right_on=['yyyymmdd', 'userid', 'impression_id'])
            print('merge 1')
            merge2 = pd.merge(merge1, df_viewable_impression_data_dump, 
                              how='left', 
                              left_on=['yyyymmdd', 'userid', 'impression_id'],
                              right_on=['yyyymmdd', 'userid', 'viewable_impression_id'])
            print('merge 2')
            merge3 = pd.merge(merge2, df_click_data_dump, 
                              how='left', 
                              left_on=['yyyymmdd', 'userid', 'response_id'],
                              right_on=['yyyymmdd', 'userid', 'click_id'])
            print('merge 3')
            merge3.to_parquet(local_extract + '/data_dump/{}_{}_data.parquet'.format(date,city), index=False)

    # folder_path = local_extract + '/data_dump'
    # parquet_files = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith('.parquet')]
    # df = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)
    
    return merge3

In [25]:
# df = func_pull_ads_data(start_date, end_date, city_list)

In [26]:
def func_percentage(df, numerator, denominator):
    df['ctr'] = (df[numerator]*100.00/df[denominator]).round(2)
    return df

<div class="alert alert-block alert-warning">
    
#### Reading Ectracted data
</div>

In [27]:
prefix_list = pd.date_range(start=start_date, end=end_date).strftime('%Y%m%d').tolist()
print(prefix_list)
path_to_directory = local_extract + '/data_dump/'

['20241128', '20241129', '20241130', '20241201', '20241202', '20241203', '20241204']


In [29]:
for prefix in prefix_list:

    print(prefix)    
    parquet_files = glob.glob(os.path.join(path_to_directory, f"{prefix}*.parquet"))
    
    for file in parquet_files:
        dataframes.append(pd.read_parquet(file))
        print(dataframes)
        

# Concatenate all DataFrames into a single DataFrame
# df_data = pd.concat(dataframes, ignore_index=True)

20241128


NameError: name 'dataframes' is not defined

In [32]:
df_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 223415017 entries, 0 to 223415016
Data columns (total 19 columns):
 #   Column                      Dtype 
---  ------                      ----- 
 0   yyyymmdd                    object
 1   city                        object
 2   os                          object
 3   responsetype                object
 4   userid                      object
 5   slotname                    object
 6   format                      object
 7   responsestatus              object
 8   taxi_ltr_segment            object
 9   taxi_retention_segment      object
 10  taxi_rr_regularity_segment  object
 11  taxi_fe_regularity_segment  object
 12  response_id                 object
 13  title                       object
 14  advertiserName              object
 15  body                        object
 16  impression_id               object
 17  viewable_impression_id      object
 18  click_id                    object
dtypes: object(19)
memory usage: 31.6+ GB


In [33]:
combined_df.shape

(223415017, 19)

In [ ]:
df = combined_df.copy()

<div class="alert alert-block alert-success">  
    
## Analysis
</div>

In [ ]:
df.head(2)

<div class="alert alert-block alert-success">  
    
#### Overall Numbers
</div>

In [20]:
list_columns = df.columns
list_columns

Index(['yyyymmdd', 'city', 'os', 'responsetype', 'userid', 'slotname',
       'format', 'responsestatus', 'isprefetched', 'taxi_ltr_segment',
       'taxi_retention_segment', 'taxi_rr_regularity_segment',
       'taxi_rr_intent_segment', 'taxi_fe_regularity_segment',
       'taxi_fe_intent_segment', 'response_id', 'title', 'advertiserName',
       'body', 'impression_id', 'viewable_impression_id', 'click_id'],
      dtype='object')

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26798401 entries, 0 to 26798400
Data columns (total 22 columns):
 #   Column                      Dtype 
---  ------                      ----- 
 0   yyyymmdd                    object
 1   city                        object
 2   os                          object
 3   responsetype                object
 4   userid                      object
 5   slotname                    object
 6   format                      object
 7   responsestatus              object
 8   isprefetched                object
 9   taxi_ltr_segment            object
 10  taxi_retention_segment      object
 11  taxi_rr_regularity_segment  object
 12  taxi_rr_intent_segment      object
 13  taxi_fe_regularity_segment  object
 14  taxi_fe_intent_segment      object
 15  response_id                 object
 16  title                       object
 17  advertiserName              object
 18  body                        object
 19  impression_id               object
 20  

In [43]:
df_overall = df_merge\
.groupby(['yyyymmdd','os', 'responsetype'])\
.agg(
    response = ('response_id', 'nunique'),
    impression = ('impression_id', 'nunique'),
    viewable_impression = ('viewable_impression_id', 'nunique'),
    click = ('click_id', 'nunique')
)\
.reset_index()

func_percentage(df_overall, 'click', 'viewable_impression')
df_overall

,yyyymmdd,os,responsetype,response,impression,viewable_impression,click,ctr
0,20241104,android,GAMBanner,79054,13806,11292,76,0.67
1,20241104,android,ONLINE_SALES,4493,4036,1806,12,0.66
2,20241104,android,System Banner,44897,38992,20485,66,0.32
3,20241104,iOS,ONLINE_SALES,888,602,130,9,6.92


In [21]:
list_columns

Index(['yyyymmdd', 'city', 'os', 'responsetype', 'userid', 'slotname',
       'format', 'responsestatus', 'isprefetched', 'taxi_ltr_segment',
       'taxi_retention_segment', 'taxi_rr_regularity_segment',
       'taxi_rr_intent_segment', 'taxi_fe_regularity_segment',
       'taxi_fe_intent_segment', 'response_id', 'title', 'advertiserName',
       'body', 'impression_id', 'viewable_impression_id', 'click_id'],
      dtype='object')

In [ ]:
source (GAM/OS/SB)
os (AND/iOS)
format (Image/Video/Story)

In [ ]:
taxi_ltr_segment (HH/PHH)
taxi_retention_segment (SGPE)
taxi_rr_regularity_segment (Daily/Weekly/Monthly)

In [ ]:
slotname(slot1/2/3)

In [ ]:
metadata-category